# Qwen 7B Fine-Tuning on Triples (Colab)

This notebook fine-tunes Qwen 7B (QLoRA) using generic triples SFT splits in Google Drive.

Expected files in Drive:
- /content/drive/MyDrive/triples/train_dev_val/sft_train.jsonl
- /content/drive/MyDrive/triples/train_dev_val/sft_dev.jsonl
- /content/drive/MyDrive/triples/train_dev_val/sft_test.jsonl

In [1]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes trl huggingface_hub sentencepiece

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import random
from dataclasses import dataclass

import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4


In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged into Hugging Face as:', whoami()['name'])
else:
    print('HF_TOKEN is missing in Colab secrets. Add it, then rerun this cell.')

In [6]:
@dataclass
class Config:
    base_model: str = 'Qwen/Qwen2.5-7B-Instruct'

    train_jsonl: str = '/content/drive/MyDrive/triples/train_dev_val/sft_train.jsonl'
    dev_jsonl: str = '/content/drive/MyDrive/triples/train_dev_val/sft_dev.jsonl'
    test_jsonl: str = '/content/drive/MyDrive/triples/train_dev_val/sft_test.jsonl'

    output_dir: str = '/content/drive/MyDrive/triples/train_dev_val/qwen7b_triples_lora'
    push_repo_id: str = 'dizza01/qwen7b-triples-lora'

    max_seq_len: int = 2048
    epochs: float = 3.0
    learning_rate: float = 2e-4
    train_batch_size: int = 1
    eval_batch_size: int = 1
    grad_accum_steps: int = 16
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0

    lora_r: int = 64
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    save_steps: int = 25
    eval_steps: int = 25
    logging_steps: int = 5

    use_4bit: bool = True

cfg = Config()
cfg

Config(base_model='Qwen/Qwen2.5-7B-Instruct', train_jsonl='/content/drive/MyDrive/triples/train_dev_val/sft_train.jsonl', dev_jsonl='/content/drive/MyDrive/triples/train_dev_val/sft_dev.jsonl', test_jsonl='/content/drive/MyDrive/triples/train_dev_val/sft_test.jsonl', output_dir='/content/drive/MyDrive/triples/train_dev_val/qwen7b_triples_lora', push_repo_id='dizza01/qwen7b-triples-lora', max_seq_len=2048, epochs=3.0, learning_rate=0.0002, train_batch_size=1, eval_batch_size=1, grad_accum_steps=16, warmup_ratio=0.03, weight_decay=0.0, lora_r=64, lora_alpha=16, lora_dropout=0.05, save_steps=25, eval_steps=25, logging_steps=5, use_4bit=True)

In [6]:
data_files = {'train': cfg.train_jsonl, 'validation': cfg.dev_jsonl}
raw = load_dataset('json', data_files=data_files)

def to_text(row):
    text = str(row.get('text', '')).strip()
    if text:
        return {'text': text}

    messages = row.get('messages', [])
    parts = []
    for m in messages:
        role = m.get('role', 'user')
        content = m.get('content', '')
        parts.append(f'<|im_start|>{role}\n{content}<|im_end|>')
    return {'text': '\n'.join(parts)}

train_text = raw['train'].map(to_text, remove_columns=raw['train'].column_names)
eval_text = raw['validation'].map(to_text, remove_columns=raw['validation'].column_names)

print('Train rows:', len(train_text))
print('Eval rows:', len(eval_text))
print(train_text[0]['text'][:700])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Train rows: 90
Eval rows: 104
<|im_start|>system
You are a QA assistant. Use only the provided context. If the answer is not present in the context, say so clearly.<|im_end|>
<|im_start|>user
Context:
Title: Characterising the effects of genetic liability to autoimmune conditions on pregnancy outcomes using Year: 2025 Source: full-text PDF tent with previous MR studies which estimated effects of systemic lupus erythematosus, rheumatoid arthritis and inflammatory bowel disease liability on selected pregnancy outcomes in FinnGen, one of our contributing cohorts(8,9,19,20). Differences in the number of autoimmune conditions and pregnancy outcomes analysed, the effects estimated, and statistical power between our study and p


In [10]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_batch(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=cfg.max_seq_len,
    )
    out['labels'] = out['input_ids'].copy()
    return out

train_tok = train_text.map(tokenize_batch, batched=True)
eval_tok = eval_text.map(tokenize_batch, batched=True)

keep = {'input_ids', 'attention_mask', 'labels'}
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep])
eval_tok = eval_tok.remove_columns([c for c in eval_tok.column_names if c not in keep])

train_tok.set_format(type='torch')
eval_tok.set_format(type='torch')

print(train_tok[0].keys())

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [11]:
if cfg.use_4bit and not torch.cuda.is_available():
    raise RuntimeError('use_4bit=True requires CUDA.')

model_kwargs = {'trust_remote_code': True}
if cfg.use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model_kwargs['quantization_config'] = bnb_config
    model_kwargs['device_map'] = 'auto'
else:
    model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(cfg.base_model, **model_kwargs)
model.config.use_cache = False

if cfg.use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 161,480,704 || all params: 7,777,097,216 || trainable%: 2.0764


In [14]:
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

# Transformers versions differ: some use eval_strategy, others evaluation_strategy.
common_args = dict(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    eval_steps=cfg.eval_steps,
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    lr_scheduler_type='cosine',
    bf16=bf16,
    fp16=fp16,
    gradient_checkpointing=True,
    report_to='none',
    dataloader_pin_memory=False,
)

try:
    training_args = TrainingArguments(
        **common_args,
        eval_strategy='steps',
    )
except TypeError:
    training_args = TrainingArguments(
        **common_args,
        evaluation_strategy='steps',
    )

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
18,1.861971,1.758077


TrainOutput(global_step=18, training_loss=2.0434546735551624, metrics={'train_runtime': 1278.9369, 'train_samples_per_second': 0.211, 'train_steps_per_second': 0.014, 'total_flos': 2.39943715651584e+16, 'train_loss': 2.0434546735551624, 'epoch': 3.0})

In [15]:
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print('Saved adapter and tokenizer to', cfg.output_dir)

Saved adapter and tokenizer to /content/drive/MyDrive/triples/train_dev_val/qwen7b_triples_lora


In [ ]:
# Push adapter + tokenizer to Hugging Face Hub
from huggingface_hub import whoami

if not cfg.push_repo_id or '/' not in cfg.push_repo_id:
    print('Set cfg.push_repo_id like "username/repo-name" and rerun.')
else:
    print('Pushing as:', whoami()['name'])
    print('Target repo:', cfg.push_repo_id)
    trainer.model.push_to_hub(cfg.push_repo_id)
    tokenizer.push_to_hub(cfg.push_repo_id)
    print('Pushed to https://huggingface.co/' + cfg.push_repo_id)

Set cfg.push_repo_id to push model to Hugging Face Hub.


In [ ]:
# Merge LoRA adapter into base model (standalone model for endpoints)
import os
import torch
from peft import AutoPeftModelForCausalLM
from huggingface_hub import HfApi

merged_dir = cfg.output_dir.rstrip('/') + '_merged'
merged_repo_id = cfg.push_repo_id + '-merged'

print('Loading adapter from:', cfg.output_dir)
model_to_merge = AutoPeftModelForCausalLM.from_pretrained(
    cfg.output_dir,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
 )

print('Merging adapter into base model...')
merged_model = model_to_merge.merge_and_unload()

os.makedirs(merged_dir, exist_ok=True)
merged_model.save_pretrained(merged_dir, safe_serialization=True, max_shard_size='5GB')
tokenizer.save_pretrained(merged_dir)

print('Merged model saved to:', merged_dir)

# Optional: push merged model to Hugging Face
push_merged = True
if push_merged:
    api = HfApi()
    api.create_repo(repo_id=merged_repo_id, repo_type='model', exist_ok=True)
    merged_model.push_to_hub(merged_repo_id, max_shard_size='5GB')
    tokenizer.push_to_hub(merged_repo_id)
    print('Merged model pushed to: https://huggingface.co/' + merged_repo_id)
else:
    print('Skipping push. Set push_merged=True to upload merged model.')